# Homework 6: Decision Trees and Ensemble Learning

Machine Learning Zoomcamp 2026 — Module 6

Dataset: `car_fuel_efficiency_2026.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

df = pd.read_csv("../datasets/car_fuel_efficiency_2026.csv")
df = df.fillna(0)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Prepare and split the dataset

Fill missing with 0, split with the exact calls given, vectorize every remaining column.

In [2]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

feature_cols = [c for c in df.columns if c != 'fuel_efficiency_mpg']

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train[feature_cols].to_dict(orient='records'))
X_val = dv.transform(df_val[feature_cols].to_dict(orient='records'))
feature_names = list(dv.get_feature_names_out())

len(feature_names), feature_names

(16,
 ['acceleration',
  'drivetrain=All-wheel drive',
  'drivetrain=Front-wheel drive',
  'drivetrain=Rear-wheel drive',
  'engine_displacement',
  'fuel_type=Diesel',
  'fuel_type=Gasoline',
  'fuel_type=Hybrid',
  'horsepower',
  'model_year',
  'num_cylinders',
  'num_doors',
  'origin=Asia',
  'origin=Europe',
  'origin=USA',
  'vehicle_weight'])

## Q1. Decision tree, max_depth=1

Which original feature is used for the root split?

In [3]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=feature_names))

split_feature = feature_names[dt.tree_.feature[0]]
split_feature.split('=')[0]

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



'model_year'

## Q2. Random forest

`n_estimators=10, random_state=1, n_jobs=-1`. RMSE on validation?

In [4]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
mean_squared_error(y_val, y_pred) ** 0.5

1.837054054730018

## Q3. Tuning n_estimators

Sweep `[10, 50, 100, 150]`, `random_state=1`. Which gives the lowest RMSE?

In [5]:
results_n = {}
for n in [10, 50, 100, 150]:
    rf_n = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    y_pred_n = rf_n.predict(X_val)
    results_n[n] = round(mean_squared_error(y_val, y_pred_n) ** 0.5, 3)

results_n

{10: 1.837, 50: 1.772, 100: 1.768, 150: 1.769}

In [6]:
min(results_n, key=results_n.get)

100

## Q4. Tuning max_depth

For each `max_depth` in `[10, 15, 20, 25]`, average RMSE across `n_estimators` in `[10, 50, 100, 150]`. Which depth wins?

In [7]:
results_depth = {}
for depth in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        rf_d = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf_d.fit(X_train, y_train)
        y_pred_d = rf_d.predict(X_val)
        rmses.append(mean_squared_error(y_val, y_pred_d) ** 0.5)
    results_depth[depth] = round(np.mean(rmses), 4)

results_depth

{10: np.float64(1.7554),
 15: np.float64(1.784),
 20: np.float64(1.7882),
 25: np.float64(1.7863)}

In [8]:
min(results_depth, key=results_depth.get)

10

## Q5. Feature importance

`n_estimators=10, max_depth=20, random_state=1`. Which of the 4 candidates is most important?

In [9]:
rf5 = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf5.fit(X_train, y_train)

importances = pd.Series(rf5.feature_importances_, index=feature_names).sort_values(ascending=False)
candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
importances[candidates]

vehicle_weight         0.203879
horsepower             0.078017
acceleration           0.055201
engine_displacement    0.061005
dtype: float64

## Q6. XGBoost eta comparison

100 rounds, same params, only `eta` changes between 0.3 and 0.1.

In [10]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names)
watchlist = [(dtrain, 'train'), (dval, 'val')]

results_eta = {}
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 0,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred = model.predict(dval)
    results_eta[eta] = mean_squared_error(y_val, y_pred) ** 0.5

results_eta

{0.3: 1.8264579799904583, 0.1: 1.724810059827976}

In [11]:
min(results_eta, key=results_eta.get)

0.1